# M4 — multi-target ATT&CK (multi-label) sobre threat

**Pregunta:** ¿cómo nos va prediciendo *técnicas* ATT&CK en vez de tiers?

**Hallazgo EDA previo (verificado en este notebook):** en `signals` live solo hay 4 técnicas: T1190 (114,507, todo threat), T1041 (7,742), T1567 (7,728 = todos los malicious), T1204 (20). suspicious = `[T1190]` (1 label), malicious = `[T1190,T1041,T1567]` (3 labels, 20 filas con +T1204). En `incident_signals` histórico aparecen 2 más: T1486 (161) y T1566 (3), casi todo `Data Theft`.

O sea: la cabeza es casi determinista del tier (T1567 ⟺ malicious) y hay **cola larga** (T1204/T1486/T1566). El experimento mide si un multi-label lineal captura la cabeza y qué pasa en la cola.

Diseño: solo filas **threat** (suspicious+malicious, 114,521 — benign no tiene técnicas y M1 ya lo filtra), targets = 6 técnicas [T1190,T1041,T1567,T1204,T1486,T1566], split **temporal 80/20** igual que M1/M2/M3, modelo `OneVsRest(LogisticRegression)` sobre TF-IDF 5k + tabular P1–P3, baseline oráculo-tier (predice técnicas desde el tier real).

In [ ]:
import json
from collections import Counter
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'pyproject.toml').exists() else ROOT.parent
print('ROOT:', ROOT)

COLS = ['label_binary','timestamp','event_time','message_sanitized','attack_techniques',
        'message_type','stream_name','product_name','vendor_name','pipeline',
        'org_id','action','severity','protocol']
df = pd.read_parquet(ROOT / 'data' / 'signals' / 'signals.parquet', columns=COLS)
th = df[df.label_binary != 'benign'].copy().reset_index(drop=True)
print('threat:', th.shape, th.label_binary.value_counts().to_dict())

TECHS = ['T1190','T1041','T1567','T1204','T1486','T1566']
def parse(v):
    try:
        t = json.loads(v) if v else []
        return [x for x in t if x in TECHS]
    except Exception:
        return []
th['tech_list'] = th.attack_techniques.fillna('').map(parse)
c = Counter(t for ts in th.tech_list for t in ts)
print('tecnicas en live-threat:', {k: c.get(k,0) for k in TECHS})
print('n-labels por evento:', th.tech_list.map(len).value_counts().to_dict())

In [ ]:
# features P1-P3 (igual que M1) + split temporal 80/20
CATS = ['message_type','stream_name','product_name','vendor_name','pipeline',
        'org_id','action','severity','protocol']
th['ts'] = pd.to_datetime(th['timestamp'], unit='s', utc=True)
th['hour'] = th.ts.dt.hour
th['msg_len'] = th.message_sanitized.fillna('').str.len()
th['lag_h'] = ((th['timestamp'] - th['event_time']) / 3600).clip(-1e4, 1e4).fillna(-1)
for col in CATS:
    th[col] = th[col].fillna('').astype(str)
msg = th.message_sanitized.fillna('')
cutoff = th['timestamp'].quantile(0.8)
tr = (th['timestamp'] <= cutoff).to_numpy(); te = ~tr
print(f'train/test: {tr.sum()}/{te.sum()}  corte: {pd.to_datetime(cutoff, unit="s", utc=True)}')

from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer(classes=TECHS)
Y = mlb.fit_transform(th.tech_list)
print('Y:', Y.shape, 'densidad:', round(Y.mean(), 4))

In [ ]:
# X: TF-IDF 5k (fit solo train) + tabular one-hot
from scipy.sparse import hstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True, min_df=5)
Ttr = vec.fit_transform(msg[tr].tolist()); Tte = vec.transform(msg[te].tolist())
tab = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30), CATS),
                           ('num', 'passthrough', ['hour','msg_len','lag_h'])])
Atr = tab.fit_transform(th[tr][CATS+['hour','msg_len','lag_h']])
Ate = tab.transform(th[te][CATS+['hour','msg_len','lag_h']])
Xtr = hstack([Ttr, Atr]).tocsr(); Xte = hstack([Tte, Ate]).tocsr()
print('X:', Xtr.shape, Xte.shape)

In [ ]:
# modelo: OneVsRest LogReg balanceado + baseline oraculo-tier
import time
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

t0 = time.time()
clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced'))
clf.fit(Xtr, Y[tr])
P = clf.predict(Xte)
print(f'fit+predict: {time.time()-t0:.0f}s')

# oraculo-tier: suspicious->[T1190], malicious->[T1190,T1041,T1567] (+T1204 impredecible)
oracle = np.zeros_like(Y[te])
lab_te = th[te].label_binary.to_numpy()
oracle[:, TECHS.index('T1190')] = 1
mal = lab_te == 'malicious'
oracle[mal, TECHS.index('T1041')] = 1
oracle[mal, TECHS.index('T1567')] = 1
print('oraculo listo')

In [ ]:
from sklearn.metrics import (classification_report, f1_score, hamming_loss,
                             accuracy_score, average_precision_score)
print('=== M4 OneVsRest-LogReg ===')
print(classification_report(Y[te], P, target_names=TECHS, zero_division=0))
print('micro-F1:', round(f1_score(Y[te], P, average='micro'), 4),
      '| macro-F1:', round(f1_score(Y[te], P, average='macro'), 4))
print('hamming:', round(hamming_loss(Y[te], P), 4),
      '| subset-acc:', round(accuracy_score(Y[te], P), 4))
print('=== baseline oraculo-tier ===')
print('micro-F1:', round(f1_score(Y[te], oracle, average='micro'), 4),
      '| macro-F1:', round(f1_score(Y[te], oracle, average='macro'), 4),
      '| hamming:', round(hamming_loss(Y[te], oracle), 4),
      '| subset-acc:', round(accuracy_score(Y[te], oracle), 4))
for j, t in enumerate(TECHS):
    try:
        ap = average_precision_score(Y[te][:, j], P[:, j])
    except Exception:
        ap = 0.0
    print(f'{t}: soporte test {int(Y[te][:, j].sum())}  PR-AUC {round(float(ap), 4)}')

## Lectura: cómo nos va con multi-target (medido)

- **T1190: P/R/F1 1.00/1.00/1.00 (PR-AUC 1.0).** Trivial: todo threat lo lleva. No mide nada, confirma el EDA.
- **T1041/T1567: P 0.22 / R 0.94-0.98 / F1 0.36, PR-AUC ~0.22.** El lineal los dispara en exceso: como suspicious es 14x malicious y comparten campos, el recall sale gratis y la precisión colapsa. M2-XGBoost (P 0.79 en malicious) separa mejor con no-linealidad + texto.
- **T1204: R 1.00 con 10 de soporte, P 0.19.** Atrapa los 10 pero a ciegas (muchos falsos positivos).
- **T1486/T1566: F1 0, sin soporte en live.** No hay de dónde aprender; el oráculo-tier también da 0 ahí (su macro es 0.4997 por eso).
- **micro-F1 0.70 vs macro-F1 0.34, hamming 0.18, subset-acc 0.43 (oráculo: 0.9994/0.4997/0.0002/0.999).** El micro lo domina T1190; el macro castiga la cola + el exceso de FPs. Reportar ambos o el paper miente.
- **Siguiente paso:** M4 como segunda cabeza sobre M2 (mismo X, otro Y) para la demo; la cola (T1204/T1486/T1566) pide few-shot/LLM con `incident_signals` histórico o reglas. No prometer ATT&CK por evento en la demo: vender cabeza, documentar cola.

In [ ]:
import json
m = {'model': 'm4-multilabel-v1', 'labels': TECHS,
     'micro_f1': round(float(f1_score(Y[te], P, average='micro')), 4),
     'macro_f1': round(float(f1_score(Y[te], P, average='macro')), 4),
     'hamming': round(float(hamming_loss(Y[te], P)), 4),
     'subset_acc': round(float(accuracy_score(Y[te], P)), 4),
     'n_train': int(tr.sum()), 'n_test': int(te.sum()),
     'oracle': {'micro_f1': round(float(f1_score(Y[te], oracle, average='micro')), 4),
                'macro_f1': round(float(f1_score(Y[te], oracle, average='macro')), 4)}}
(ROOT / 'models' / 'm4_metrics.json').write_text(json.dumps(m, indent=2))
print('guardado models/m4_metrics.json', m)